In [80]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import statsmodels.api as sm
import numpy as np

from scipy.stats import ttest_ind
from pycircstat2 import Circular
from pycircstat2.hypothesis import wheeler_watson_test
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests


In [3]:
headache_df = pd.read_csv('Migrebot_26.10.2026_a773a1b49713.csv')
meteo_df = pd.read_csv('open-meteo-41.58N41.69E6m.csv', skiprows=3)

In [4]:
headache_df.head()

,Дата,Время,Головная боль,Признаки мигрени,Принятые медикаменты,Интенсивность боли,Локализация,Характер,Нагрузки,Тошнота,ФоТофобия,ФоНофобия,Триггеры,Начало боли,Окончание боли,Комментарии,Свой вопрос
0,2026-10-25,21:16,Нет,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-10-24,21:15,Нет,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-10-23,21:10,Да,NaN,"Ибупрофен 400, Немного помогло",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-10-22,21:40,Нет,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-10-21,21:58,Нет,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
meteo_df.head()

,time,weather_code (wmo code),temperature_2m_mean (°C),apparent_temperature_mean (°C),wind_direction_10m_dominant (°),shortwave_radiation_sum (MJ/m²),et0_fao_evapotranspiration (mm),sunrise (iso8601),sunset (iso8601),daylight_duration (s),...,dew_point_2m_mean (°C),et0_fao_evapotranspiration_sum (mm),relative_humidity_2m_mean (%),snowfall_water_equivalent_sum (mm),pressure_msl_mean (hPa),surface_pressure_mean (hPa),winddirection_10m_dominant (°),wind_gusts_10m_mean (km/h),wind_speed_10m_mean (km/h),wet_bulb_temperature_2m_mean (°C)
0,2026-05-03,51,10.1,9.4,344,10.63,1.46,2026-05-03T06:07,2026-05-03T20:13,50710.10,...,9.4,1.46,96,0.0,1013.7,1012.9,344,13.1,3.6,9.6
1,2026-05-04,63,11.0,10.2,187,10.63,1.51,2026-05-04T06:06,2026-05-04T20:14,50848.77,...,10.0,1.51,94,0.0,1007.7,1007.0,187,19.3,5.3,10.3
2,2026-05-05,53,11.8,11.2,278,11.14,1.59,2026-05-05T06:05,2026-05-05T20:15,50986.51,...,10.7,1.59,93,0.0,1012.7,1011.9,278,16.0,5.6,11.0
3,2026-05-06,53,12.4,12.0,259,12.62,1.83,2026-05-06T06:04,2026-05-06T20:16,51123.20,...,11.1,1.83,92,0.0,1018.0,1017.3,259,15.2,5.4,11.5
4,2026-05-07,3,12.8,12.4,323,20.37,2.98,2026-05-07T06:02,2026-05-07T20:17,51258.72,...,10.3,2.98,85,0.0,1021.3,1020.5,323,13.4,3.8,11.1


In [6]:
headache_df.shape

(156, 17)

In [7]:
meteo_df.shape

(176, 26)

In [8]:
headache_df['Дата'] = pd.to_datetime(headache_df['Дата'])

In [9]:
meteo_df['time'] = pd.to_datetime(meteo_df['time'])

In [10]:
analysis_df = meteo_df.merge(headache_df[['Дата', 'Головная боль']], how='left', left_on='time', right_on='Дата')

In [11]:
# Missing diary entries were coded as no headache (0), based on the participant’s confirmation that all headache days were recorded.
analysis_df['Головная боль'] = (
    analysis_df['Головная боль'].fillna('Нет')
)

In [12]:
analysis_df = analysis_df.drop(columns='Дата')

In [13]:
analysis_df['headache'] = analysis_df['Головная боль'].map({
    'Да':1,
    'Нет':0
})

In [14]:
analysis_df = analysis_df.drop(columns='Головная боль')

In [15]:
analysis_df = analysis_df.sort_values('time').copy()

analysis_df['pressure_previous_day'] = (
    analysis_df['pressure_msl_mean (hPa)'].shift(1)
)

In [16]:
analysis_df['pressure_change'] = (
    analysis_df['pressure_msl_mean (hPa)']
    - analysis_df['pressure_previous_day']
)

In [17]:
analysis_df.head()

,time,weather_code (wmo code),temperature_2m_mean (°C),apparent_temperature_mean (°C),wind_direction_10m_dominant (°),shortwave_radiation_sum (MJ/m²),et0_fao_evapotranspiration (mm),sunrise (iso8601),sunset (iso8601),daylight_duration (s),...,snowfall_water_equivalent_sum (mm),pressure_msl_mean (hPa),surface_pressure_mean (hPa),winddirection_10m_dominant (°),wind_gusts_10m_mean (km/h),wind_speed_10m_mean (km/h),wet_bulb_temperature_2m_mean (°C),headache,pressure_previous_day,pressure_change
0,2026-05-03,51,10.1,9.4,344,10.63,1.46,2026-05-03T06:07,2026-05-03T20:13,50710.10,...,0.0,1013.7,1012.9,344,13.1,3.6,9.6,1,NaN,NaN
1,2026-05-04,63,11.0,10.2,187,10.63,1.51,2026-05-04T06:06,2026-05-04T20:14,50848.77,...,0.0,1007.7,1007.0,187,19.3,5.3,10.3,0,1013.7,-6.0
2,2026-05-05,53,11.8,11.2,278,11.14,1.59,2026-05-05T06:05,2026-05-05T20:15,50986.51,...,0.0,1012.7,1011.9,278,16.0,5.6,11.0,0,1007.7,5.0
3,2026-05-06,53,12.4,12.0,259,12.62,1.83,2026-05-06T06:04,2026-05-06T20:16,51123.20,...,0.0,1018.0,1017.3,259,15.2,5.4,11.5,1,1012.7,5.3
4,2026-05-07,3,12.8,12.4,323,20.37,2.98,2026-05-07T06:02,2026-05-07T20:17,51258.72,...,0.0,1021.3,1020.5,323,13.4,3.8,11.1,1,1018.0,3.3


In [18]:
analysis_df.shape

(176, 29)

In [20]:
analysis_df['headache'].value_counts()


headache
0    138
1     38
Name: count, dtype: int64

In [21]:
# Доли дней с головной болью и без нее
analysis_df['headache'].value_counts(normalize=True) * 100

headache
0    78.409091
1    21.590909
Name: proportion, dtype: float64

In [22]:
exclude = [
    'headache',
    'weather_code (wmo code)',
    'wind_direction_10m_dominant (°)',
    'winddirection_10m_dominant (°)'
]

weather_columns = (
    analysis_df
    .select_dtypes(include='number')
    .columns
    .drop(exclude, errors='ignore')
)

summary = (
    analysis_df
    .groupby('headache')[weather_columns]
    .agg(['count', 'mean', 'median', 'std'])
    .T
    .rename(columns={0: 'No headache', 1: 'Headache'})
    .round(2)
)

with pd.option_context(
    'display.max_rows', None,
    'display.max_columns', None
):
    display(summary)

headache                                    No headache  Headache
temperature_2m_mean (°C)            count        138.00     38.00
                                    mean          20.73     19.27
                                    median        21.60     20.60
                                    std            2.98      3.40
apparent_temperature_mean (°C)      count        138.00     38.00
                                    mean          23.45     21.10
                                    median        24.15     22.60
                                    std            4.37      4.92
shortwave_radiation_sum (MJ/m²)     count        138.00     38.00
                                    mean          16.85     16.72
                                    median        16.70     14.74
                                    std            6.54      7.72
et0_fao_evapotranspiration (mm)     count        138.00     38.00
                                    mean           3.02      2.97
                                    median         3.06      2.58
                                    std            1.15      1.39
daylight_duration (s)               count        138.00     38.00
                                    mean       49628.36  48981.42
                                    median     51543.07  51008.88
                                    std         4912.43   5346.70
sunshine_duration (s)               count        138.00     38.00
                                    mean       35114.30  31756.60
                                    median     38506.80  35700.95
                                    std        14311.65  16264.88
precipitation_sum (mm)              count        138.00     38.00
                                    mean           7.14      7.69
                                    median         2.25      2.15
                                    std           11.05     14.98
rain_sum (mm)                       count        138.00     38.00
                                    mean           7.14      7.69
                                    median         2.25      2.15
                                    std           11.05     14.98
snowfall_sum (cm)                   count        138.00     38.00
                                    mean           0.00      0.00
                                    median         0.00      0.00
                                    std            0.00      0.00
precipitation_hours (h)             count        138.00     38.00
                                    mean           9.23      8.45
                                    median         9.00      7.00
                                    std            7.54      7.98
cloud_cover_mean (%)                count        138.00     38.00
                                    mean          59.85     58.66
                                    median        60.00     68.00
                                    std           26.60     33.53
dew_point_2m_mean (°C)              count        138.00     38.00
                                    mean          18.51     16.39
                                    median        19.25     17.20
                                    std            3.19      4.07
et0_fao_evapotranspiration_sum (mm) count        138.00     38.00
                                    mean           3.02      2.97
                                    median         3.06      2.58
                                    std            1.15      1.39
relative_humidity_2m_mean (%)       count        138.00     38.00
                                    mean          87.73     84.53
                                    median        89.00     87.00
                                    std            6.15      9.81
snowfall_water_equivalent_sum (mm)  count        138.00     38.00
                                    mean           0.00      0.00
                                    median         0.00      0.00
                                    std 

# Research question #

Is there an association between daily weather conditions and the occurrence of headaches?

**Significance level**: α = 0.05.

*This is an exploratory analysis of one person's daily observations. Statistical inference must account for possible dependence between consecutive days. Multiple-testing adjustment will be applied across the weather hypotheses tested.*

In [46]:
analysis_df.columns

Index(['time', 'weather_code (wmo code)', 'temperature_2m_mean (°C)',
       'apparent_temperature_mean (°C)', 'wind_direction_10m_dominant (°)',
       'shortwave_radiation_sum (MJ/m²)', 'et0_fao_evapotranspiration (mm)',
       'sunrise (iso8601)', 'sunset (iso8601)', 'daylight_duration (s)',
       'sunshine_duration (s)', 'precipitation_sum (mm)', 'rain_sum (mm)',
       'snowfall_sum (cm)', 'precipitation_hours (h)', 'cloud_cover_mean (%)',
       'dew_point_2m_mean (°C)', 'et0_fao_evapotranspiration_sum (mm)',
       'relative_humidity_2m_mean (%)', 'snowfall_water_equivalent_sum (mm)',
       'pressure_msl_mean (hPa)', 'surface_pressure_mean (hPa)',
       'winddirection_10m_dominant (°)', 'wind_gusts_10m_mean (km/h)',
       'wind_speed_10m_mean (km/h)', 'wet_bulb_temperature_2m_mean (°C)',
       'headache', 'pressure_previous_day', 'pressure_change'],
      dtype='str')

In [61]:
test_results = {}

## Weather variables and headaches: exploratory comparisons

We compare quantitative weather variables between days with and without headaches.

For each weather variable, we test the following hypotheses:

- **H₀:** The population mean is the same on headache and non-headache days.
- **H₁:** The population mean differs between headache and non-headache days.

### Method

- Test: Two-sided Welch t-test.
- Significance level: α = 0.05.
- Mean difference: Mean on headache days − mean on non-headache days.
- Missing values are excluded separately for each variable.
- Holm adjustment will be applied across all tested weather hypotheses.

Weather codes, wind direction, constant variables, and duplicate columns are excluded from this set of t-tests. Wind direction is analysed separately using a circular test.

### Limitations

These are preliminary comparisons. Welch t-tests assume independent observations, whereas consecutive days may be related. Temporal dependence and seasonal differences are not accounted for in these tests, and Holm adjustment does not resolve these limitations.

The results describe associations, not causal effects. A non-significant result does not establish the absence of an association.

In [62]:
weather_columns = [
    'temperature_2m_mean (°C)',
    'apparent_temperature_mean (°C)',
    'shortwave_radiation_sum (MJ/m²)',
    'et0_fao_evapotranspiration (mm)',
    'daylight_duration (s)',
    'sunshine_duration (s)',
    'precipitation_sum (mm)',
    'precipitation_hours (h)',
    'cloud_cover_mean (%)',
    'dew_point_2m_mean (°C)',
    'relative_humidity_2m_mean (%)',
    'pressure_msl_mean (hPa)',
    'wind_gusts_10m_mean (km/h)',
    'wind_speed_10m_mean (km/h)',
    'wet_bulb_temperature_2m_mean (°C)'
]
for column in weather_columns:

    with_headache = analysis_df.loc[
        analysis_df['headache'] == 1, column
    ].dropna()

    without_headache = analysis_df.loc[
        analysis_df['headache'] == 0, column
    ].dropna()

    if min(len(with_headache), len(without_headache)) < 2:
        print(f'Skipped: {column} — insufficient observations')
        continue

    if with_headache.var() == 0 and without_headache.var() == 0:
        print(f'Skipped: {column} — zero variance in both groups')
        continue

    result = ttest_ind(
        with_headache,
        without_headache,
        equal_var=False,
        alternative='two-sided'
    )

    test_results[column] = {
        'test': 'Welch t-test',
        'n_headache': len(with_headache),
        'n_no_headache': len(without_headache),
        'mean_headache': with_headache.mean(),
        'mean_no_headache': without_headache.mean(),
        'mean_difference': (
            with_headache.mean() - without_headache.mean()
        ),
        't_statistic': result.statistic,
        'p_value_raw': result.pvalue
    }

## Wind direction and headaches

*We compare the distribution of daily dominant wind direction between days with and without headaches.*

H₀: The distribution of daily dominant wind direction is the same on headache and non-headache days.

H₁: The distribution of daily dominant wind direction differs between headache and non-headache days.

*Note: Wind direction is a circular variable. A standard Welch t-test is not appropriate for this comparison.*

In [64]:
column = 'wind_direction_10m_dominant (°)'

wind_with_headache = analysis_df.loc[
    analysis_df['headache'] == 1, column
].dropna().to_numpy()

wind_without_headache = analysis_df.loc[
    analysis_df['headache'] == 0, column
].dropna().to_numpy()

if min(len(wind_with_headache), len(wind_without_headache)) < 10:
    raise ValueError('At least 10 observations per group are needed here.')

circular_headache = Circular(
    wind_with_headache,
    unit='degree'
)

circular_no_headache = Circular(
    wind_without_headache,
    unit='degree'
)
wind_result = wheeler_watson_test(
    samples=[circular_headache, circular_no_headache],
    verbose=True
)
print(type(wind_result))
print(wind_result)
test_results[column] = {
    'test': 'Wheeler–Watson',
    'n_headache': len(wind_with_headache),
    'n_no_headache': len(wind_without_headache),
    'w_statistic': wind_result.W,
    'p_value_raw': wind_result.pval
}

The Wheeler and Watson Two/Multi-Sample Test
---------------------------------------------
H0: All samples are from populations with the same angle.
HA: All samples are not from populations with the same angle.

Test Statistics: 27.12927
P-value: 0.00000 ***
<class 'pycircstat2.hypothesis.WheelerWatsonTestResult'>
WheelerWatsonTestResult(W=27.1292701792941, pval=1.2851500324694748e-06, df=2)


## Weather type and headaches

*We compare the proportion of headache days across three weather categories: clear/cloudy, drizzle, and rain.*

- **H₀:** Weather type and headache occurrence are independent.
- **H₁:** Weather type and headache occurrence are associated.

### Method

- Test: Pearson’s chi-square test of independence.
- Significance level: α = 0.05.
- All expected cell counts exceed 5.
- The raw p-value will be included in the Holm adjustment across the tested weather hypotheses.

### Preliminary results

The observed proportions of headache days were:

- Clear/cloudy: 12/43 — **27.9%**.
- Drizzle: 15/75 — **20.0%**.
- Rain: 11/58 — **19.0%**.

### Limitations

The test assumes independent observations and does not account for possible dependence between consecutive days or seasonal differences. A non-significant result does not prove that no association exists.

In [66]:
analysis_df['weather_code (wmo code)'].value_counts().sort_index()

weather_code (wmo code)
0      3
1      5
2      6
3     29
51    36
53    31
55     8
61    22
63    33
65     3
Name: count, dtype: int64

In [67]:
weather_groups = {
    0: 'Clear / cloudy',
    1: 'Clear / cloudy',
    2: 'Clear / cloudy',
    3: 'Clear / cloudy',
    51: 'Drizzle',
    53: 'Drizzle',
    55: 'Drizzle',
    61: 'Rain',
    63: 'Rain',
    65: 'Rain'
}

analysis_df['weather_group'] = (
    analysis_df['weather_code (wmo code)'].map(weather_groups)
)

weather_table = pd.crosstab(
    analysis_df['weather_group'],
    analysis_df['headache']
)

display(weather_table)

headache,0,1
weather_group,,
Clear / cloudy,31,12
Drizzle,60,15
Rain,47,11


In [69]:
chi_result = chi2_contingency(
    weather_table,
    correction=False
)

print('Chi-square statistic:', chi_result.statistic)
print('Degrees of freedom:', chi_result.dof)
print('Raw p-value:', chi_result.pvalue)

test_results['weather_group'] = {
    'test': 'Chi-square',
    'n_headache': int(weather_table[1].sum()),
    'n_no_headache': int(weather_table[0].sum()),
    'chi2_statistic': chi_result.statistic,
    'p_value_raw': chi_result.pvalue
}

Chi-square statistic: 1.3615419952202614
Degrees of freedom: 2
Raw p-value: 0.5062265424125559


In [70]:
test_results_df = (
    pd.DataFrame.from_dict(test_results, orient='index')
    .rename_axis('weather_metric')
    .reset_index()
)

display(test_results_df.round(4))

,weather_metric,test,n_headache,n_no_headache,mean_headache,mean_no_headache,mean_difference,t_statistic,p_value_raw,w_statistic,chi2_statistic
0,temperature_2m_mean (°C),Welch t-test,38,138,19.2684,20.7290,-1.4606,-2.4045,0.0197,NaN,NaN
1,apparent_temperature_mean (°C),Welch t-test,38,138,21.1000,23.4507,-2.3507,-2.6690,0.0100,NaN,NaN
2,shortwave_radiation_sum (MJ/m²),Welch t-test,38,138,16.7218,16.8541,-0.1322,-0.0965,0.9235,NaN,NaN
3,et0_fao_evapotranspiration (mm),Welch t-test,38,138,2.9679,3.0156,-0.0477,-0.1937,0.8471,NaN,NaN
4,daylight_duration (s),Welch t-test,38,138,48981.4150,49628.3638,-646.9488,-0.6719,0.5045,NaN,NaN
5,sunshine_duration (s),Welch t-test,38,138,31756.5955,35114.2966,-3357.7011,-1.1554,0.2530,NaN,NaN
6,precipitation_sum (mm),Welch t-test,38,138,7.6921,7.1384,0.5537,0.2125,0.8326,NaN,NaN
7,precipitation_hours (h),Welch t-test,38,138,8.4474,9.2319,-0.7845,-0.5430,0.5893,NaN,NaN
8,cloud_cover_mean (%),Welch t-test,38,138,58.6579,59.8478,-1.1899,-0.2019,0.8408,NaN,NaN
9,dew_point_2m_mean (°C),Welch t-test,38,138,16.3868,18.5065,-2.1197,-2.9709,0.0045,NaN,NaN


In [72]:
test_results_df = (
    pd.DataFrame.from_dict(test_results, orient='index')
    .rename_axis('weather_metric')
    .reset_index()
)

if not test_results_df['p_value_raw'].between(0, 1).all():
    raise ValueError('Check missing or invalid p-values.')

reject, p_adjusted, _, _ = multipletests(
    test_results_df['p_value_raw'],
    alpha=0.05,
    method='holm'
)

test_results_df['p_value_holm'] = p_adjusted
test_results_df['reject_h0_holm'] = reject

display(
    test_results_df[
        ['weather_metric', 'test', 'p_value_raw',
         'p_value_holm', 'reject_h0_holm']
    ].sort_values('p_value_holm')
)

,weather_metric,test,p_value_raw,p_value_holm,reject_h0_holm
15,wind_direction_10m_dominant (°),Wheeler–Watson,0.000001,0.000022,True
9,dew_point_2m_mean (°C),Welch t-test,0.004547,0.072756,False
14,wet_bulb_temperature_2m_mean (°C),Welch t-test,0.006833,0.102489,False
1,apparent_temperature_mean (°C),Welch t-test,0.010019,0.140262,False
0,temperature_2m_mean (°C),Welch t-test,0.019680,0.255837,False
10,relative_humidity_2m_mean (%),Welch t-test,0.061970,0.743636,False
13,wind_speed_10m_mean (km/h),Welch t-test,0.104395,1.000000,False
12,wind_gusts_10m_mean (km/h),Welch t-test,0.126854,1.000000,False
11,pressure_msl_mean (hPa),Welch t-test,0.281329,1.000000,False
8,cloud_cover_mean (%),Welch t-test,0.840769,1.000000,False


In [73]:
wind_column = 'wind_direction_10m_dominant (°)'
angles = analysis_df[wind_column]

if not angles.dropna().between(0, 360).all():
    raise ValueError('Wind direction must be between 0 and 360 degrees.')
analysis_df['wind_sector'] = pd.cut(
    (angles + 22.5) % 360,
    bins=range(0, 361, 45),
    labels=['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'],
    right=False
)

display(
    analysis_df[[wind_column, 'wind_sector']].head(10)
)

,wind_direction_10m_dominant (°),wind_sector
0,344,N
1,187,S
2,278,W
3,259,W
4,323,NW
5,208,SW
6,171,S
7,211,SW
8,235,SW
9,249,W


In [75]:
direction_column = 'wind_sector'  
headache_column = 'headache'   

counts = pd.crosstab(
    index=analysis_df[direction_column],
    columns=analysis_df[headache_column]
)

shares = counts.div(counts.sum(axis=0), axis='columns') * 100

wind_summary = pd.concat(
    {
        'Days': counts,
        'Share (%)': shares
    },
    axis='columns'
)

display(wind_summary)

Days      Share (%)           
headache       0   1          0          1
wind_sector                               
N              4   1   2.898551   2.631579
NE             2   0   1.449275   0.000000
E              3   1   2.173913   2.631579
SE             7   5   5.072464  13.157895
S             10   7   7.246377  18.421053
SW            36  10  26.086957  26.315789
W             55   9  39.855072  23.684211
NW            21   5  15.217391  13.157895

In [82]:
wind_model_df = (
    analysis_df[
        ['time', 'headache', 'wind_direction_10m_dominant (°)']
    ]
    .copy()
    .rename(columns={
        'wind_direction_10m_dominant (°)': 'wind_degrees'
    })
    .sort_values('time')
)

if wind_model_df.isna().any().any():
    raise ValueError('Check missing values before fitting the model.')

if not wind_model_df['headache'].isin([0, 1]).all():
    raise ValueError('Headache must contain only 0 and 1.')

if not wind_model_df['wind_degrees'].between(0, 360).all():
    raise ValueError('Wind direction must be between 0 and 360.')

if not wind_model_df['time'].diff().dropna().eq(
    pd.Timedelta(days=1)
).all():
    raise ValueError('Check duplicate dates or gaps in the calendar.')


angles = np.deg2rad(wind_model_df['wind_degrees'] % 360)

wind_model_df['wind_sin'] = np.sin(angles)
wind_model_df['wind_cos'] = np.cos(angles)


days = (
    wind_model_df['time'] - wind_model_df['time'].min()
).dt.days

wind_model_df['time_trend'] = (days - days.mean()) / 30
wind_model_df['time_trend_squared'] = (
    wind_model_df['time_trend'] ** 2
)

X = sm.add_constant(
    wind_model_df[
        ['wind_sin', 'wind_cos',
         'time_trend', 'time_trend_squared']
    ]
)

y = wind_model_df['headache']

wind_models = {}
wind_check_rows = []


for lag in [3, 7, 14]:

    fitted_model = sm.GLM(
        y,
        X,
        family=sm.families.Binomial()
    ).fit(
        cov_type='HAC',
        cov_kwds={
            'maxlags': lag,
            'use_correction': True
        }
    )

    if not fitted_model.converged:
        raise RuntimeError(f'Model did not converge: HAC lag {lag}')

    wind_models[lag] = fitted_model


    joint_test = fitted_model.wald_test(
        'wind_sin = 0, wind_cos = 0',
        use_f=False,
        scalar=True
    )

    wind_check_rows.append({
        'hac_lag': lag,
        'n_days': int(fitted_model.nobs),
        'wald_statistic': float(joint_test.statistic),
        'df': 2,
        'p_value_raw': float(joint_test.pvalue)
    })

wind_check_df = pd.DataFrame(wind_check_rows)

display(wind_check_df)

,hac_lag,n_days,wald_statistic,df,p_value_raw
0,3,176,3.737852,2,0.154289
1,7,176,3.172028,2,0.204740
2,14,176,3.283907,2,0.193602


## Wind direction: follow-up analysis

The initial difference in wind direction distributions remained statistically significant after Holm adjustment.

However, a follow-up logistic regression using sine and cosine terms for wind direction, a quadratic time trend, and HAC standard errors did not provide sufficient evidence of an association (joint Wald test, p = 0.205 at the primary HAC lag of 7 days). The conclusion was unchanged with HAC lags of 3 and 14 days.

The two analyses test different hypotheses under different assumptions. Therefore, the change in results cannot be attributed solely to adjustment for time.

Overall, the evidence remains inconclusive. These results do not establish wind direction as a headache-related factor, but they do not demonstrate the absence of an association.